Test Cases for Fluxonium circuit :
* anHarmonicity flatness across flux spectrum
* analytically related to Ej

In [1]:
from des_scq import models
from des_scq.optimization import Optimization
from des_scq.circuit import Charge
from des_scq.discovery import lossAnharmonicity,lossTransitionFlatness
from torch import tensor
from numpy import arange,linspace,array
from des_scq.utils import plotCompare
from torch import set_num_threads
set_num_threads(32)

* initial circuit complies with oscillator basis
* spectrum is invariant with higher truncation

In [2]:
basis = {'O':[512],'I':[],'J':[]}
basis = [512]
circuit = models.fluxonium(Charge,basis,El=.1,Ec=1.,Ej=50)
print(circuit.circuitComponents())

{'Cap': 0.5, 'JJ': 50.000000000000014, 'I': 0.25330295910584444}


In [3]:
flux_range = tensor(linspace(0,1,31,endpoint=True))

#### pre-optimization Fluxonium

In [4]:
flux_profile = [[flux] for flux in flux_range]
flux_point = ('I')

In [5]:
H_LC = circuit.hamiltonianLC()
H_J = circuit.hamiltonianJosephson

In [6]:
Spectrum = circuit.spectrumManifold(flux_profile)
pre_Ex = []
for energies,_ in Spectrum:
    energies = energies-energies[0]
    pre_Ex.append(energies.detach().numpy())
pre_Ex = array(pre_Ex).T

In [7]:
plotCompare(flux_range,{'Ist':pre_Ex[1],'IInd':pre_Ex[2],'anharmonicity':pre_Ex[2]-2*pre_Ex[1]},
            'Excitation Profile : pre-optimization','external flux','energy(GHz)',export='pdf',size=(600,800))

#### optimization

In [8]:
optim = Optimization(circuit,flux_profile,lossTransitionFlatness)
dLogs,dParams,dCircuit = optim.optimization(50)

In [9]:
plotCompare(dLogs.index,dLogs[['loss']],'Optimization-Flux Sensitivity','iteration',export='pdf',size=(600,800))

In [10]:
plotCompare(dCircuit.index,dCircuit,'Optimization-Circuit Parameters',"iteration","energy(GHz)",export='pdf',size=(600,800))

Analytically the solution is $JJ=0$ \
However, the optimization stagnates asymptotically \
and therefore gradient descent algorithm improper

#### post-optimization Fluxonium

In [11]:
dCircuit.iloc[0]

Cap     1.0
JJ     50.0
I       0.1
Name: 0, dtype: float64

In [12]:
dCircuit.iloc[-1]

Cap     0.952432
JJ     52.388538
I       0.095239
Name: 50, dtype: float64

In [13]:
Spectrum = circuit.spectrumManifold(flux_profile)
post_Ex = []
for energies,_ in Spectrum:
    energies = energies-energies[0]
    post_Ex.append(energies.detach().numpy())
post_Ex = array(post_Ex).T

In [14]:
plotCompare(flux_range,{'Ist':post_Ex[1],'IInd':post_Ex[2],'anharmonicity':post_Ex[2]-2*post_Ex[1]},
            'Excitation Profile : post-optimization','external flux','energy(GHz)',export='pdf',size=(600,800))

In [15]:
print(circuit.circuitComponents())

{'Cap': 0.5254898094564756, 'JJ': 52.43796379619142, 'I': 0.26622850795075237}


## Circuit Comparison

In [16]:
plotCompare(flux_range,{'post':post_Ex[2]-2*post_Ex[1],'pre':pre_Ex[2]-2*pre_Ex[1]},
            'Flux Profile-Anharmonicity','external flux','anharmonicity(GHz)',export='pdf',size=(600,800))